In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Simple grid search for hyperparameters
    """
    # Define hyperparameter grid
    """{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.5, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 100, 'early_stopping_patience': 15}
"""
    param_grid = [
        {
            'hidden_dims': [512, 256, 128, 64],
            'dropout_rate': 0.4,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 64,
            'num_epochs': 100,
            'early_stopping_patience': 15
        },
        {
            'hidden_dims': [512, 256, 128],
            'dropout_rate': 0.3,
            'learning_rate': 0.0005,
            'weight_decay': 1e-4,
            'batch_size': 64,
            'num_epochs': 100,
            'early_stopping_patience': 15
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 100,
            'early_stopping_patience': 15
        }
    ]
    
    best_config = None
    best_score = 0
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    print(f"\n{'='*60}")
    print("Best configuration:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config

# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 1: Use best known configuration
    best_config = {
        'hidden_dims': [512, 256, 128, 64],
        'dropout_rate': 0.4,
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'batch_size': 64,
        'num_epochs': 100,
        'early_stopping_patience': 15
    }
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    
    # Option 3: Cross-validation (uncomment to use)
    cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v5.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()

Using device: cpu

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/3
{'hidden_dims': [512, 256, 128, 64], 'dropout_rate': 0.4, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 64, 'num_epochs': 100, 'early_stopping_patience': 15}
Epoch 1/100
  Train Loss: 5.4946, Train F1: 0.3256
  Val Loss: 1.5238, Val F1: 0.4975
  LR: 0.001000
  ✓ New best F1: 0.4975
  ✓ New best F1: 0.6186
  ✓ New best F1: 0.6477
  ✓ New best F1: 0.7235
Epoch 5/100
  Train Loss: 1.4731, Train F1: 0.5418
  Val Loss: 0.4154, Val F1: 0.7212
  LR: 0.001000
Epoch 10/100
  Train Loss: 0.7130, Train F1: 0.6803
  Val Loss: 0.5129, Val F1: 0.7067
  LR: 0.000500
  → Learning rate reduced from 0.001000 to 0.000500
  ✓ New best F1: 0.7257
Epoch 15/100
  Train Loss: 0.5155, Train F1: 0.7391
  Val Loss: 0.5651, Val F1: 0.7022
  LR: 0.000500
Epoch 20/100
  Train Loss: 0.4517, Train F1: 0.7711
  Val Loss: 0.5898, Val F1: 0.6893
  LR: 0.000250
Epoch 25/100
  Train L

In [4]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Simple grid search for hyperparameters
    """
    # Define hyperparameter grid - Optimized around best config [768, 384, 192]
    param_grid = [
        # Original best config
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Try deeper architecture with similar pattern
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Try wider architecture
        {
            'hidden_dims': [896, 448, 224],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with lower dropout
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.45,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with higher dropout
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with lower learning rate
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.0005,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with higher learning rate
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.0015,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with more weight decay
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 5e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with larger batch size
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 64,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Alternative architecture: symmetric reduction
        {
            'hidden_dims': [640, 320, 160],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        }
    ]
    
    best_config = None
    best_score = 0
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    print(f"\n{'='*60}")
    print("Best configuration:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config

# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 1: Use best known configuration
    """best_config = {
        'hidden_dims': [512, 256, 128, 64],
        'dropout_rate': 0.4,
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'batch_size': 64,
        'num_epochs': 100,
        'early_stopping_patience': 15
    }"""
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    
    # Option 3: Cross-validation (uncomment to use)
    cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v6.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()

Using device: cpu

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/10
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.5, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 2.7128, Train F1: 0.5203
  Val Loss: 0.7257, Val F1: 0.6657
  LR: 0.001000
  ✓ New best F1: 0.6657
  ✓ New best F1: 0.7212
Epoch 5/120
  Train Loss: 1.1421, Train F1: 0.6999
  Val Loss: 1.0467, Val F1: 0.6724
  LR: 0.001000
Epoch 10/120
  Train Loss: 0.7618, Train F1: 0.8001
  Val Loss: 1.0340, Val F1: 0.6191
  LR: 0.000500
Epoch 15/120
  Train Loss: 0.3930, Train F1: 0.8613
  Val Loss: 1.1717, Val F1: 0.6229
  LR: 0.000250
Epoch 20/120
  Train Loss: 0.2477, Train F1: 0.8821
  Val Loss: 1.2637, Val F1: 0.6029
  LR: 0.000125
  → Learning rate reduced from 0.000250 to 0.000125

Early stopping triggered after 22 epochs

Training completed. Best Val F1: 0.7212

Final Validat

In [5]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Simple grid search for hyperparameters
    """
    # Define hyperparameter grid - Optimized around best config [768, 384, 192]
    param_grid = [
        # Original best config
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Try deeper architecture with similar pattern
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Try wider architecture
        {
            'hidden_dims': [896, 448, 224],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with lower dropout
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.45,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with higher dropout
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with lower learning rate
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.0005,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with higher learning rate
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.0015,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with more weight decay
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 5e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Best config with larger batch size
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 64,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Alternative architecture: symmetric reduction
        {
            'hidden_dims': [640, 320, 160],
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        }
    ]
    
    best_config = None
    best_score = 0
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    print(f"\n{'='*60}")
    print("Best configuration:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config

# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 1: Use best known configuration
    """best_config = {
        'hidden_dims': [512, 256, 128, 64],
        'dropout_rate': 0.4,
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'batch_size': 64,
        'num_epochs': 100,
        'early_stopping_patience': 15
    }"""
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    print("\nBest Parameters Found:")
    print(best_config)

    
    # Option 3: Cross-validation (uncomment to use)
    #cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v6.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()

Using device: cpu

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/10
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.5, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 2.7128, Train F1: 0.5203
  Val Loss: 0.7257, Val F1: 0.6657
  LR: 0.001000
  ✓ New best F1: 0.6657
  ✓ New best F1: 0.7212
Epoch 5/120
  Train Loss: 1.1421, Train F1: 0.6999
  Val Loss: 1.0467, Val F1: 0.6724
  LR: 0.001000
Epoch 10/120
  Train Loss: 0.7618, Train F1: 0.8001
  Val Loss: 1.0340, Val F1: 0.6191
  LR: 0.000500
Epoch 15/120
  Train Loss: 0.3930, Train F1: 0.8613
  Val Loss: 1.1717, Val F1: 0.6229
  LR: 0.000250
Epoch 20/120
  Train Loss: 0.2477, Train F1: 0.8821
  Val Loss: 1.2637, Val F1: 0.6029
  LR: 0.000125
  → Learning rate reduced from 0.000250 to 0.000125

Early stopping triggered after 22 epochs

Training completed. Best Val F1: 0.7212

Final Validat